K-Prototype Clustering

Alur: Data Mentah → Pra-Processing → Normalize/Filter → K-Prototype → Hasil Excel


  Install library Python

In [ ]:
! pip install pandas numpy matplotlib seaborn kmodes openpyxl

  import library yang digunakan

In [3]:
import pandas as pd
import numpy as np
import re
from scipy.stats import chi2_contingency
from datetime import datetime
import warnings
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
warnings.filterwarnings('ignore')

PRA-PROCESSING

In [4]:
# ─────────────────────────────────────────────
# LOAD DATA MENTAH
# ─────────────────────────────────────────────
INPUT_FILE  ='/content/drive/MyDrive/KULIAH/Semester 8/SKRIPSI-TugasAkhir/SKRIPSI /PENGOLAHAN DATA/SK TRAYEK.xlsx'
OUTPUT_FILE = '/content/drive/MyDrive/KULIAH/Semester 8/SKRIPSI-TugasAkhir/SKRIPSI /PENGOLAHAN DATA/hasil_clustering_kprototype.xlsx'
print("=" * 60)
print(" K-PROTOTYPE CLUSTERING - SK TRAYEK")
print("=" * 60)

df_raw = pd.read_excel(INPUT_FILE)
print(f"\n DATA MENTAH  : {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")


# ─────────────────────────────────────────────
# Seleksi 3 kolom
# ─────────────────────────────────────────────
cols_needed = ['Alamat Pemilik', 'Kode Trayek', 'BERAKHIR SK TRAYEK']
df = df_raw[cols_needed].copy()
df.columns = ['alamat', 'kode_trayek', 'berakhir_sk']

# ─────────────────────────────────────────────
# Hapus Baris Kosong dengan NaN
# ─────────────────────────────────────────────
before = len(df)
df.dropna(subset=['alamat', 'kode_trayek', 'berakhir_sk'], inplace=True)
print(f"   Baris setelah drop NaN: {len(df)} (buang {before - len(df)} baris)")

 K-PROTOTYPE CLUSTERING - SK TRAYEK

 DATA MENTAH  : 6017 baris, 8 kolom
   Baris setelah drop NaN: 6017 (buang 0 baris)


NORMALIZE/FILTER DATA

In [5]:
# ─────────────────────────────────────────────
# NORMALIZE / FILTER DATA
# ─────────────────────────────────────────────

KECAMATAN_PATTERNS = [
    'BABAKAN MADANG', 'BOJONG GEDE', 'CARINGIN', 'CARIU', 'CIAMPEA', 'CIAWI',
    'CIBINONG', 'CIBUNGBULANG', 'CIGUDEG', 'CIGOMBONG', 'CIJERUK',
    'CILEUNGSI', 'CIOMAS', 'CISARUA', 'CISEENG', 'CITEUREUP', 'DRAMAGA',
    'GUNUNG PUTRI', 'GUNUNG SINDUR', 'JASINGA', 'JONGGOL', 'KEMANG',
    'KLAPANUNGGAL', 'LEUWILIANG', 'LEUWISADENG', 'MEGAMENDUNG', 'NANGGUNG',
    'PAMIJAHAN', 'PARUNG', 'RANCABUNGUR', 'RUMPIN', 'SUKAMAKMUR', 'SUKARAJA',
    'TAJUR HALANG', 'TAMANSARI', 'TANJUNGSARI', 'TENJOLAYA' ,

]

# Kamus revisi: "Kata Salah" : "KATA BENAR"
# Tambahkan variasi typo lain yang ditemukan di Excel ke dalam dictionary ini
kecamatan_revision = {
    "1": "GUNUNG PUTRI",
    "10": "GUNUNG PUTRI",
    "10JONGGOL": "JONGGOL",
    "12": "GUNUNG PUTRI",
    "13": "GUNUNG PUTRI",
    "15": "CIBINONG",
    "1CIAMPEA": "CIAMPEA",
    "1CILEUNGSI": "CILEUNGSI",
    "1CILEUNGSI BOGOR": "CILEUNGSI",
    "1DRAMAGA": "DRAMAGA",
    "1PAMIJAHAN": "PAMIJAHAN",
    "1PARUNG": "PARUNG",
    "28": "CILEUNGSI",
    "2CITUERUP": "CITEUREUP",
    "2RUMPIN": "RUMPIN",
    "4": "GUNUNG PUTRI",
    "5CILEUNGSI": "CILEUNGSI",
    "7": "SUKARAJA",
    "7JASINGA": "JASINGA",
    "7JONGGOL": "JONGGOL",
    "BABAKANMADANG": "BABAKAN MADANG",
    "BARANG": "TAMANSARI",
    "BBK.MADANG BGR": "BABAKAN MADANG",
    "BBK.MADANG BOGOR": "BABAKAN MADANG",
    "MANDANG": "BABAKAN MADANG",
    "RT.6/3BBK.MADANG BOGOR": "BABAKAN MADANG",
    "BJ.GEDE": "BOJONG GEDE",
    "BJ.GEDE BOGOR": "BOJONG GEDE",
    "BJGEDE": "BOJONG GEDE",
    "BOJONG GEDE": "BOJONG GEDE",
    "BOJONGEDE BOGOR": "BOJONG GEDE",
    "BOJONGGEDE": "BOJONG GEDE",
    "BOJONGGEDE BOGOR": "BOJONG GEDE",
    "CB.BULANG BGR": "CIBUNGBULANG",
    "CB.BULANG BOGOR": "CIBUNGBULANG",
    "CBBULANG BOGOR": "CIBUNGBULANG",
    "CIAMPA BOGOR": "CIAMPEA",
    "CIBINOPNG BOGOR": "CIBINONG",
    "CIBUNBULAN": "CIBUNGBULANG",
    "CIBUNGBULAN": "CIBUNGBULANG",
    "CIBUNGBULANG BOGOR": "CIBUNGBULANG",
    "CIBUNGBULANGG": "CIBUNGBULANG",
    "CIGIDEG": "CIGUDEG",
    "CIJERUK BOGOR": "CIJERUK",
    "CIJUJUNG": "SUKARAJA",
    "CILENGSI": "CILEUNGSI",
    "CILEUNGSI BGR": "CILEUNGSI",
    "CIMANDALA SUKARAJA BOGOR": "SUKARAJA",
    "CIMANGGIS": "BOJONG GEDE",
    "CIMANGGIS BOGOR": "BOJONG GEDE",
    "CIRTEUREUP BOGOR": "CITEUREUP",
    "CITEREUP BOGOR": "CITEUREUP",
    "CITERUP": "CITEUREUP",
    "CITEUEREUP BOGOR": "CITEUREUP",
    "CITEUREP": "CITEUREUP",
    "CITEUREUP BOGOR": "CITEUREUP",
    "CITEUREUPBOGOR": "CITEUREUP",
    "CIUDEG": "CIGUDEG",
    "CUTEUREUP BOGOR": "CITEUREUP",
    "DARMAGA BOGOR": "DRAMAGA",
    "DRAMAG BOGOR": "DRAMAGA",
    "DRAMAGA BOGOR": "DRAMAGA",
    "DRAMGA BGR": "DRAMAGA",
    "GN PUTRI": "GUNUNG PUTRI",
    "GN. PUTRI": "GUNUNG PUTRI",
    "GN.PUT RI BOGOR": "GUNUNG PUTRI",
    "GN.PUTRI BGR": "GUNUNG PUTRI",
    "GN.PUTRI BOGOR": "GUNUNG PUTRI",
    "GN.PUTRI KAB.BOGOR": "GUNUNG PUTRI",
    "GN.PUTRI SELATAN": "GUNUNG PUTRI",
    "GN.PUTRI UTARA": "GUNUNG PUTRI",
    "GN.SINDUR BOGOR": "GUNUNG SINDUR",
    "GRY.CIPEUCANG INDAH": "CILEUNGSI",
    "GUNUNGPUTRI": "GUNUNG PUTRI",
    "GUNUNGPUTRI BGR": "GUNUNG PUTRI",
    "HALANG": "TAJUR HALANG",
    "JONGGL BOGOR": "JONGGOL",
    "JONGGOL BOGOR": "JONGGOL",
    "K.NUNGGAL BGR": "KLAPANUNGGAL",
    "KALISUREN": "TAJUR HALANG",
    "KALISUREN BOGOR": "TAJUR HALANG",
    "KD.HALANG BOGOR": "SUKARAJA",
    "KEC.CIJERUK BOGOR": "CIJERUK",
    "KEDUNG HALANG": "TAJUR HALANG",
    "KLAPANU": "KLAPANUNGGAL",
    "KLAPANUNGGAKL BOGOR": "KLAPANUNGGAL",
    "KLAPANUNGGALNGGAKL": "KLAPANUNGGAL",
    "KLAPANUNGGAL BOGOR": "KLAPANUNGGAL",
    "KLAPANUNGGALNGGAL": "KLAPANUNGGAL",
    "KLP.NINGGAL BOGOR": "KLAPANUNGGAL",
    "KLP.NUNGGAL BOGOR": "KLAPANUNGGAL",
    "KLP.NUNGGAL BGR": "KLAPANUNGGAL",
    "KLPANUNGGAL BOGOR": "KLAPANUNGGAL",
    "KLAPANUNGGAKL BOGOR": "KLAPANUNGGAL",
    "KLAPAKLAPANUNGGAL": "KLAPANUNGGAL",
    "KP.CIKALAGAN": "CILEUNGSI",
    "KP.TEGALWARURT.2/1CIAMPEA BOGOR":"CIAMPEA",
    "LAYA": "TENJOLAYA",
    "LIANG": "LEUWILIANG",
    "LW.LIANG BGR": "LEUWILIANG",
    "LW.LIANG BOGOR": "LEUWILIANG",
    "LEUWILIANG BOGOR": "LEUWILIANG",
    "LEUWILEUWILIANG": "LEUWILIANG",
    "LW.SADENG BOGOR": "LEUWISADENG",
    "LIMO": "PARUNG",
    "MEGAMENDUNG BOGOR": "MEGAMENDUNG",
    "MG.MENDUNG BGR": "MEGAMENDUNG",
    "MG.MENDUNG BOGOR": "MEGAMENDUNG",
    "MAKMUR":"SUKAMAKMUR",
    "MADANG": "BABAKAN MADANG",
    "PAMIJAHN BOGOR": "PAMIJAHAN",
    "PANIJAHAN BOGOR": "PAMIJAHAN",
    "PAMIJHAN BOGOR": "PAMIJAHAN",
    "PARUNG PANJANG": "PARUNGPANJANG",
    "PUTRICIAMPEA": "CIAMPEA",
    "PUTRICILEUNGSI": "CILEUNGSI",
    "PUTRICITEUREUP": "CITEUREUP",
    "PUTRIDRAMAGA": "DRAMAGA",
    "PUTRIPARUNG": "PUTRI PARUNG",
    "RC.BUNGUR BOGOR": "RANCABUNGUR",
    "RT.2/1CIAMPEA BOGOR": "CIAMPEA",
    "RT.6/1CIAMPEA BOGOR": "CIAMPEA",
    "RT.3/1CILEUNGSI BOGOR": "CILEUNGSI",
    "RT.3/1CILEUNGSI BGR": "CILEUNGSI",
    "RT.2/1CITEUREUP BOGOR": "CITEUREUP",
    "RT.2/1DRAMAGA BOGOR": "DRAMAGA",
    "RT.2/1PARUNG BOGOR": "PARUNG",
    "RT.2/1GN.SINDUR BOGOR": "GUNUNG SINDUR",
    "RT.2/3 GN. SINDUR BOGOR": "GUNUNG SINDUR",
    "RT.6/3BBK.MADANG BOGOR": "BABAKAN MADANG",
    "SADENG LW. SADENG BOGOR": "LEUWISADENG",
    "SEMPLAK": "KEMANG",
    "SUKAJADI BOGOR": "TAMANSARI",
    "SINDUR": "GUNUNG SINDUR",
    "SK.MAKMUR BOGOR": "SUKAMAKMUR",
    "SUKASUKAMAKMUR": "SUKAMAKMUR",
    "TAJUR HALANG BOGOR": "TAJUR HALANG",
    "TAJURHALANG BOGOR": "TAJUR HALANG",
    "TJ.HALANG": "TAJUR HALANG",
    "TJ.HALANG BOGOR": "TAJUR HALANG",
    "TJR.HALANG BGR": "TAJUR HALANG",
    "TJR.HALANG BOGOR": "TAJUR HALANG",
    "TONJONG BOGOR": "TAJUR HALANG",
    "TNJ.LAYA BOGOR": "TENJOLAYA",
    "TENJOLAYA BOGOR": "TENJOLAYA",
    "TENJOTENJOLAYA": "TENJOLAYA",
    "NUNGGAL": "KLAPANUNGGAL",
    "WANAHERANG GN.PUTRI": "GUNUNG PUTRI",
}

KECAMATAN_PATTERNS_SORTED = sorted(KECAMATAN_PATTERNS, key=len, reverse=True)
KECAMATAN_REGEX = '|'.join(
    r'\b' + re.escape(k) + r'\b' for k in KECAMATAN_PATTERNS_SORTED
)

def ekstrak_kecamatan(alamat):
    if not isinstance(alamat, str):
        return 'TIDAK DIKETAHUI'

    alamat_upper = alamat.upper()

    # 1. PERBAIKI TYPO TERLEBIH DAHULU
    # Harus dilakukan awal agar typo di akhir kalimat (misal "CITERUP")
    # berubah menjadi kata baku ("CITEUREUP") yang dikenali Regex.
    for typo, perbaikan in kecamatan_revision.items():
        alamat_upper = alamat_upper.replace(typo.upper(), perbaikan.upper())

    # 2. CARI SEMUA KECAMATAN LALU AMBIL YANG PALING AKHIR (Ide Utama Anda)
    # Gunakan re.findall untuk menemukan semua kecocokan, bukan re.search
    matches = re.findall(KECAMATAN_REGEX, alamat_upper)

    if matches:
        # matches[-1] mengambil elemen list yang paling belakang / terakhir
        return matches[-1].strip()

    # 3. FALLBACK 1: Cari kata sebelum BOGOR di akhir kalimat
    fallback = re.search(r'(\w+)\s+(?:KAB\.?\s*)?BOGOR', alamat_upper)
    if fallback:
        # Jika berhasil dapat, pastikan kita cek lagi siapa tahu masih ada typo
        hasil_fallback = fallback.group(1).strip()
        return kecamatan_revision.get(hasil_fallback, hasil_fallback)

    # 4. FALLBACK 2: Jika semua gagal, ambil kata paling ujung
    tokens = re.findall(r'[A-Z]+', alamat_upper)
    if tokens:
        kata_terakhir = tokens[-1]
        # Cek kamus typo sekali lagi untuk kata terakhir ini
        return kecamatan_revision.get(kata_terakhir, kata_terakhir)

    return 'TIDAK DIKETAHUI'

def to_unix_timestamp(tanggal):
    if pd.isna(tanggal):
        return 0
    if isinstance(tanggal, (int, float)):
        return int(tanggal)
    dt = pd.to_datetime(tanggal, errors='coerce')
    if pd.isna(dt):
        return 0
    return int(dt.timestamp())


print("\n[3] NORMALIZE / FILTER DATA ...")

def normalize(df_in, label=''):
    df_out = df_in.copy()

    # Ekstraksi dan perbaikan wilayah
    df_out['kecamatan'] = df_out['alamat'].apply(ekstrak_kecamatan)

    # Transformasi waktu
    df_out['berakhir_sk_unix'] = df_out['berakhir_sk'].apply(to_unix_timestamp)

    # Standarisasi kolom referensi
    df_out['kode_trayek'] = df_out['kode_trayek'].astype(str).str.strip().str.upper()

    # Filter data valid
    sebelum = len(df_out)
    df_out = df_out[df_out['berakhir_sk_unix'] > 0]
    df_out = df_out[df_out['kecamatan'] != 'TIDAK DIKETAHUI']

    print(f"   [{label}] Setelah filter: {len(df_out)} baris (buang {sebelum - len(df_out)})")
    return df_out.reset_index(drop=True)


# Proses pertama
df_norm1 = normalize(df, label='Proses 1')

# Proses kedua - validasi ulang
print("\n   Mengulang proses untuk memastikan data tidak halu ...")
df_norm2 = normalize(df_norm1[['alamat', 'kode_trayek', 'berakhir_sk']], label='Proses 2')

df_clean = df_norm2.copy()
print(f"\n   Total data bersih siap clustering: {len(df_clean)} baris")


[3] NORMALIZE / FILTER DATA ...
   [Proses 1] Setelah filter: 6017 baris (buang 0)

   Mengulang proses untuk memastikan data tidak halu ...
   [Proses 2] Setelah filter: 6017 baris (buang 0)

   Total data bersih siap clustering: 6017 baris


ALGORITMA K-PROTOTYPE

In [6]:
# ─────────────────────────────────────────────
# 4. K-PROTOTYPE (implementasi dari scratch)
# ─────────────────────────────────────────────

class KPrototypes:
    def __init__(self, n_clusters=4, max_iter=100, gamma=0.5, n_init=5, random_state=42):
        self.n_clusters   = n_clusters
        self.max_iter     = max_iter
        self.gamma        = gamma
        self.n_init       = n_init
        self.random_state = random_state

    def _distance(self, x_num, x_cat, c_num, c_cat):
        num_dist = np.sum((x_num - c_num) ** 2)
        cat_dist = np.sum(x_cat != c_cat)
        return num_dist + self.gamma * cat_dist

    def _assign(self, X_num, X_cat, c_num, c_cat):
        labels = np.zeros(len(X_num), dtype=int)
        for i in range(len(X_num)):
            dists = [self._distance(X_num[i], X_cat[i], c_num[k], c_cat[k])
                     for k in range(self.n_clusters)]
            labels[i] = np.argmin(dists)
        return labels

    def _update(self, X_num, X_cat, labels):
        c_num = np.zeros((self.n_clusters, X_num.shape[1]))
        c_cat = np.empty((self.n_clusters, X_cat.shape[1]), dtype=object)
        for k in range(self.n_clusters):
            mask = labels == k
            if mask.sum() == 0:
                idx = np.random.randint(len(X_num))
                c_num[k] = X_num[idx]
                c_cat[k] = X_cat[idx]
                continue
            c_num[k] = X_num[mask].mean(axis=0)
            for j in range(X_cat.shape[1]):
                vals, counts = np.unique(X_cat[mask, j], return_counts=True)
                c_cat[k, j] = vals[np.argmax(counts)]
        return c_num, c_cat

    def _cost(self, X_num, X_cat, labels, c_num, c_cat):
        total = 0.0
        for i in range(len(X_num)):
            total += self._distance(X_num[i], X_cat[i], c_num[labels[i]], c_cat[labels[i]])
        return total

    def fit(self, X_num, X_cat):
        np.random.seed(self.random_state)
        best_labels, best_cost = None, np.inf
        best_cn, best_cc = None, None
        for _ in range(self.n_init):
            idxs = np.random.choice(len(X_num), self.n_clusters, replace=False)
            c_num = X_num[idxs].copy().astype(float)
            c_cat = X_cat[idxs].copy()
            labels = None
            for _ in range(self.max_iter):
                new_labels = self._assign(X_num, X_cat, c_num, c_cat)
                c_num, c_cat = self._update(X_num, X_cat, new_labels)
                if labels is not None and np.array_equal(labels, new_labels):
                    break
                labels = new_labels
            cost = self._cost(X_num, X_cat, labels, c_num, c_cat)
            if cost < best_cost:
                best_cost = cost
                best_labels = labels.copy()
                best_cn, best_cc = c_num.copy(), c_cat.copy()
        self.labels_ = best_labels
        self.centers_num_ = best_cn
        self.centers_cat_ = best_cc
        self.cost_ = best_cost
        return self

# ─────────────────────────────────────────────────────────────────
#  ─────────── 4a Mencari K optimal (Elbow Method)  ───────────────
# ─────────────────────────────────────────────────────────────────

def pilih_k_optimal(X_num, X_cat, k_range=range(2, 8)):
    print("\n[4a] Mencari K optimal (Elbow Method) ...")
    costs = {}
    for k in k_range:
        m = KPrototypes(n_clusters=k, n_init=3, random_state=42)
        m.fit(X_num, X_cat)
        costs[k] = m.cost_
        print(f"     K={k}  cost={costs[k]:.2f}")
    ks   = sorted(costs.keys())
    vals = [costs[k] for k in ks]
    diffs = [vals[i] - vals[i+1] for i in range(len(vals)-1)]
    k_opt = ks[int(np.argmax(diffs)) + 1]
    if k_opt < 2:
        k_opt = 3
    print(f"\n   K optimal terpilih: {k_opt}")
    return k_opt, costs


X_num = df_clean[['berakhir_sk_unix']].values.astype(float)
X_cat = df_clean[['kecamatan', 'kode_trayek']].values.astype(str)
X_num_std = (X_num - X_num.min()) / (X_num.max() - X_num.min() + 1e-9)

print("\n[4] ALGORITMA K-PROTOTYPE ...")
K_OPT, elbow_costs = pilih_k_optimal(X_num_std, X_cat, k_range=range(2, 8))

# ─────────────────────────────────────────────────────────────────
#   ──────────────── 4b Training model final  ────────────────────
# ─────────────────────────────────────────────────────────────────

print(f"\n[4b] Training model final K={K_OPT} (n_init=10) ...")
model = KPrototypes(n_clusters=K_OPT, n_init=10, random_state=42)
model.fit(X_num_std, X_cat)
df_clean['cluster'] = model.labels_ + 1

# ─────────────────────────────────────────────────────────────────
#   ────── 4c EVALUASI: PENGARUH FITUR (Cramér's V & Eta²) ──────────
# Tujuan: mengukur seberapa besar kontribusi masing-masing fitur
# dalam membentuk pemisahan antar cluster yang terbentuk.
#
# Cramér's V  → digunakan untuk fitur KATEGORIKAL (Kecamatan, Kode Trayek)
#   Cara kerja: mengukur seberapa tidak merata distribusi kategori
#   antar cluster via uji Chi-Square. Nilai 0 = tidak ada asosiasi,
#   nilai 1 = asosiasi sempurna (tiap kategori eksklusif di satu cluster).
#
# Eta²        → digunakan untuk fitur NUMERIK (Berakhir SK / Unix Timestamp)
#   Cara kerja: mengukur proporsi variansi total nilai numerik yang
#   dapat dijelaskan oleh perbedaan antar cluster (between-cluster variance
#   dibagi total variance). Nilai mendekati 1 = cluster sangat memisahkan
#   nilai numerik tersebut.
#
# Kedua ukuran ini dipilih karena sesuai dengan tipe datanya masing-masing
# dan keduanya memiliki rentang nilai 0–1 sehingga mudah dibandingkan.
# ─────────────────────────────────────────────────────────────────

print("\n[4c] Pengaruh Fitur Terhadap Cluster ...")

labels_all = df_clean['cluster'].values
n_total    = len(labels_all)

# ── Cramér's V: Kecamatan ──────────────────────────────────────────
# Chi-Square menghitung apakah distribusi kecamatan berbeda secara
# signifikan antar cluster. Cramér's V menormalisasinya ke 0–1.
ct_kec              = pd.crosstab(labels_all, df_clean['kecamatan'])
chi2_kec, _, _, _   = chi2_contingency(ct_kec)
k_kec               = min(ct_kec.shape) - 1
cramers_kec         = float(np.sqrt(chi2_kec / (n_total * k_kec))) if k_kec > 0 else 0.0

# ── Cramér's V: Kode Trayek ────────────────────────────────────────
ct_tray             = pd.crosstab(labels_all, df_clean['kode_trayek'])
chi2_tray, _, _, _  = chi2_contingency(ct_tray)
k_tray              = min(ct_tray.shape) - 1
cramers_tray        = float(np.sqrt(chi2_tray / (n_total * k_tray))) if k_tray > 0 else 0.0

# ── Eta²: Unix Timestamp (Berakhir SK) ────────────────────────────
# Between-cluster variance: seberapa jauh rata-rata tiap cluster
# dari rata-rata keseluruhan (dikali bobot jumlah anggotanya)
overall_mean  = df_clean['berakhir_sk_unix'].mean()
between_var   = sum(
    len(df_clean[df_clean['cluster'] == c]) *
    (df_clean[df_clean['cluster'] == c]['berakhir_sk_unix'].mean() - overall_mean) ** 2
    for c in df_clean['cluster'].unique()
)
total_var     = ((df_clean['berakhir_sk_unix'] - overall_mean) ** 2).sum()
eta2          = float(between_var / total_var) if total_var > 0 else 0.0

# ── Tentukan fitur paling berpengaruh ─────────────────────────────
skor_fitur = {
    'Kode Trayek' : cramers_tray,
    'Kecamatan'   : cramers_kec,
    'Berakhir SK' : eta2,
}
fitur_terkuat = max(skor_fitur, key=skor_fitur.get)

print(f"   Kode Trayek  → Cramér's V : {cramers_tray:.4f}")
print(f"   Kecamatan    → Cramér's V : {cramers_kec:.4f}")
print(f"   Berakhir SK  → Eta²       : {eta2:.4f}")
print(f"\n   Fitur paling berpengaruh  : {fitur_terkuat} ({skor_fitur[fitur_terkuat]:.4f})")


summary = df_clean.groupby('cluster').agg(
    jumlah=('cluster', 'count'),
    kecamatan_dominan=('kecamatan', lambda x: x.value_counts().index[0]),
    kode_trayek_dominan=('kode_trayek', lambda x: x.value_counts().index[0]),
    rata_rata_berakhir=('berakhir_sk', 'mean'),
    min_berakhir=('berakhir_sk', 'min'),
    max_berakhir=('berakhir_sk', 'max'),
).reset_index()

print("\n[5] RINGKASAN CLUSTER :")
print(summary.to_string(index=False))



[4] ALGORITMA K-PROTOTYPE ...

[4a] Mencari K optimal (Elbow Method) ...
     K=2  cost=5034.24
     K=3  cost=4646.90
     K=4  cost=4483.84
     K=5  cost=4141.65
     K=6  cost=3891.52
     K=7  cost=3641.87

   K optimal terpilih: 3

[4b] Training model final K=3 (n_init=10) ...

[4c] Pengaruh Fitur Terhadap Cluster ...
   Kode Trayek  → Cramér's V : 0.6260
   Kecamatan    → Cramér's V : 0.5837
   Berakhir SK  → Eta²       : 0.2529

   Fitur paling berpengaruh  : Kode Trayek (0.6260)

[5] RINGKASAN CLUSTER :
 cluster  jumlah kecamatan_dominan kode_trayek_dominan            rata_rata_berakhir min_berakhir max_berakhir
       1    2230      GUNUNG PUTRI                 05B 2024-09-08 22:46:23.139013376   2014-04-04   2034-02-05
       2    2453          CIBINONG                 05C 2020-07-06 00:20:32.776192512   2007-06-01   2030-10-13
       3    1334         CILEUNGSI                  64 2021-07-24 08:01:26.356821504   2010-02-15   2030-05-26


EXPORT EXCEL

In [7]:
# ─────────────────────────────────────────────
# 5. EXPORT KE EXCEL
# ─────────────────────────────────────────────
print("\n[6] EXPORT KE EXCEL ...")

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

HDR_FILL = PatternFill("solid", fgColor="1F4E79")
HDR_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=10)
DATA_FONT = Font(name="Arial", size=10)
CLUSTER_COLORS = ["B2AC88", "A2CFFE", "F5F5DC", "98FF98", "B0C4DE", "F4A460", "E2C1A4", "8FBC8F"]
thin = Side(style='thin', color='CCCCCC')
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)


def write_header(ws, headers, row=1):
    for c, h in enumerate(headers, 1):
        cell = ws.cell(row, c, h)
        cell.fill = HDR_FILL
        cell.font = HDR_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = BORDER
    ws.row_dimensions[row].height = 30


# ────────────────────── Sheet 1: Hasil Clustering ──────────────────────
ws1 = wb.active
ws1.title = "Hasil Clustering"

df_export = df_clean[['alamat', 'kecamatan', 'kode_trayek', 'berakhir_sk', 'berakhir_sk_unix', 'cluster']].copy()
df_export.columns = ['Alamat Asli', 'Kecamatan', 'Kode Trayek', 'Berakhir SK', 'Unix Timestamp', 'Cluster']
df_export['Berakhir SK'] = pd.to_datetime(df_export['Berakhir SK']).dt.strftime('%Y-%m-%d')
df_export = df_export.sort_values('Cluster').reset_index(drop=True)

write_header(ws1, list(df_export.columns))
for r, row in enumerate(df_export.itertuples(index=False), 2):
    cl = row[-1]
    rfill = PatternFill("solid", fgColor=CLUSTER_COLORS[(cl-1) % len(CLUSTER_COLORS)] + "33")
    for c, val in enumerate(row, 1):
        cell = ws1.cell(r, c, val)
        cell.font = DATA_FONT
        cell.border = BORDER
        cell.fill = rfill
        if c == len(df_export.columns):
            cell.alignment = Alignment(horizontal='center')
            cell.font = Font(name="Arial", size=10, bold=True)

for i, w in enumerate([35, 22, 12, 14, 18, 8], 1):
    ws1.column_dimensions[get_column_letter(i)].width = w
ws1.freeze_panes = "A2"

#───────────SHEET 2: RINGKASAN CLUSTER ───────────
ws2 = wb.create_sheet("Ringkasan Cluster")

#───────────RINGKASAN CLUSTER DOMINAN───────────#
# Definisi STATUS_FILL & STATUS_FONT diletakkan di sini agar bisa dipakai Sheet 2 dan seterusnya
STATUS_FILL = {
    'BERAKHIR': PatternFill("solid", fgColor="C00000"),
    'RENTAN':   PatternFill("solid", fgColor="FF8C00"),
    'PANJANG':  PatternFill("solid", fgColor="375623"),
}
STATUS_FONT = {
    'BERAKHIR': Font(name="Arial", size=10, bold=True, color="FFFFFF"),
    'RENTAN':   Font(name="Arial", size=10, bold=True, color="FFFFFF"),
    'PANJANG':  Font(name="Arial", size=10, bold=True, color="FFFFFF"),
}
STATUS_ROW_FILL = {
    'BERAKHIR': PatternFill("solid", fgColor="FFF0F0"),
    'RENTAN':   PatternFill("solid", fgColor="FFF5E6"),
    'PANJANG':  PatternFill("solid", fgColor="F0FFF0"),
}

# Buat status_map dari cluster → status berdasarkan rata-rata berakhir_sk_unix
# Cluster dengan berakhir_sk paling awal = BERAKHIR, tengah = RENTAN, paling akhir = PANJANG
_cm = df_clean.groupby('cluster')['berakhir_sk_unix'].mean().sort_values()
_urut = list(_cm.index)
_n = len(_urut); _t = max(_n // 3, 1)
status_map = {}
for _i, _cl in enumerate(_urut):
    if _i < _t:           status_map[_cl] = 'BERAKHIR'
    elif _i < 2 * _t:     status_map[_cl] = 'RENTAN'
    else:                  status_map[_cl] = 'PANJANG'

sum_headers = ['Cluster', 'Jumlah Kendaraan', 'Kecamatan Dominan',
               'Kode Trayek Dominan', 'Rata-rata Berakhir SK',
               'Min Berakhir SK', 'Max Berakhir SK', 'Status SK']  # [+1 kolom]
write_header(ws2, sum_headers)
for r, row in enumerate(summary.itertuples(index=False), 2):
    cl     = row[0]
    status = status_map.get(cl, '')
    c_fill = PatternFill("solid", fgColor=CLUSTER_COLORS[(cl-1) % len(CLUSTER_COLORS)])
    for c, val in enumerate(row, 1):
        if isinstance(val, pd.Timestamp):
            val = val.strftime('%Y-%m-%d')
        cell = ws2.cell(r, c, val)
        cell.border = BORDER
        cell.alignment = Alignment(horizontal='center' if c in [1,2] else 'left')
        if c == 1:
            cell.fill = c_fill
            cell.font = Font(name="Arial", size=10, bold=True, color="000000")
        else:
            cell.font = DATA_FONT
    # Tulis kolom Status SK (kolom ke-8, setelah 7 kolom summary)
    cell_status = ws2.cell(r, 8, status)
    cell_status.fill      = STATUS_FILL.get(status, PatternFill("solid", fgColor="808080"))
    cell_status.font      = STATUS_FONT.get(status, DATA_FONT)
    cell_status.border    = BORDER
    cell_status.alignment = Alignment(horizontal='center', vertical='center')

for i, w in enumerate([10, 18, 25, 22, 22, 18, 18, 12], 1):  # +12 untuk Status SK
    ws2.column_dimensions[get_column_letter(i)].width = w

#───────────List semua kecamatan beserta cluster-nya───────────#
# Hitung ringkasan per kecamatan: cluster dominan, jumlah kendaraan, rata-rata berakhir SK
kec_summary = df_clean.groupby('kecamatan').agg(
    cluster_dominan=('cluster', lambda x: x.value_counts().index[0]),
    trayek_dominan=('kode_trayek',   lambda x: x.value_counts().index[0]),
    jumlah_kendaraan=('kecamatan', 'count'),
    rata_rata_berakhir=('berakhir_sk', 'mean'),
    min_berakhir=('berakhir_sk', 'min'),
    max_berakhir=('berakhir_sk', 'max'),
).reset_index().sort_values(['cluster_dominan', 'kecamatan']).reset_index(drop=True)

# Baris pemisah antara Bagian A dan Bagian B
baris_pisah = len(summary) + 3
ws2.merge_cells(f'A{baris_pisah}:H{baris_pisah}')
sep_cell = ws2.cell(baris_pisah, 1, 'DAFTAR KECAMATAN PER CLUSTER')
sep_cell.font      = Font(name="Arial", size=10, bold=True, color="FFFFFF")
sep_cell.fill      = PatternFill("solid", fgColor="2E4057")
sep_cell.alignment = Alignment(horizontal='center', vertical='center')
sep_cell.border    = BORDER
ws2.row_dimensions[baris_pisah].height = 22

# Header tabel kecamatan
kec_headers = ['No', 'Kecamatan', 'Cluster', 'Trayek Dominan', 'Jumlah Kendaraan',
               'Rata-rata Berakhir SK', 'Min Berakhir SK', 'Max Berakhir SK',
               'Status SK']  # [+1 kolom]
write_header(ws2, kec_headers, row=baris_pisah + 1)

# Isi baris kecamatan
for i, row_kec in enumerate(kec_summary.itertuples(index=False), 1):
    r_excel = baris_pisah + 1 + i
    cl      = row_kec[1]  # cluster_dominan
    trayek  = row_kec[2]  # trayek_dominan
    status  = status_map.get(cl, '')  # [TAMBAHAN] ambil status dari cluster dominan
    c_fill  = PatternFill("solid", fgColor=CLUSTER_COLORS[(cl - 1) % len(CLUSTER_COLORS)] + "44")
    vals = [
        i,
        row_kec[0],                                                                        # kecamatan
        cl,                                                                                 # cluster_dominan
        trayek,                                                                             # trayek_dominan
        row_kec[3],                                                                         # jumlah_kendaraan
        row_kec[4].strftime('%Y-%m-%d') if isinstance(row_kec[4], pd.Timestamp) else '',  # rata_rata_berakhir
        row_kec[5].strftime('%Y-%m-%d') if isinstance(row_kec[5], pd.Timestamp) else '',  # min_berakhir
        row_kec[6].strftime('%Y-%m-%d') if isinstance(row_kec[6], pd.Timestamp) else '',  # max_berakhir
        status,                                                                             # status_sk [TAMBAHAN]
    ]
    for c, val in enumerate(vals, 1):
        cell = ws2.cell(r_excel, c, val)
        cell.border    = BORDER
        cell.font      = DATA_FONT
        cell.fill      = c_fill
        cell.alignment = Alignment(
            horizontal='left' if c == 2 else 'center',
            vertical='center'
        )
        if c == 3:   # kolom Cluster — warna solid cluster
            cell.fill = PatternFill("solid", fgColor=CLUSTER_COLORS[(cl - 1) % len(CLUSTER_COLORS)])
            cell.font = Font(name="Arial", size=10, bold=True, color="000000")
        elif c == 9:  # kolom Status SK — warna status [TAMBAHAN]
            cell.fill = STATUS_FILL.get(status, PatternFill("solid", fgColor="808080"))
            cell.font = STATUS_FONT.get(status, DATA_FONT)
    ws2.row_dimensions[r_excel].height = 18

for i, w in enumerate([5, 22, 10, 16, 14, 18, 14, 14, 12], 1):  # +12 untuk Status SK
    ws2.column_dimensions[get_column_letter(i)].width = w
ws2.freeze_panes = "A2"

# ── Sheet 3: Distribusi Kecamatan ───────────
ws3 = wb.create_sheet("Distribusi Kecamatan")
pivot = df_clean.groupby(['kecamatan', 'cluster']).size().unstack(fill_value=0).reset_index()
pivot.columns = ['Kecamatan'] + [f'Cluster {c}' for c in pivot.columns[1:]]
pivot['Total'] = pivot.iloc[:, 1:].sum(axis=1)
pivot = pivot.sort_values('Total', ascending=False)

write_header(ws3, list(pivot.columns))
for r, row in enumerate(pivot.itertuples(index=False), 2):
    alt_fill = PatternFill("solid", fgColor="EEF4FF") if r % 2 == 0 else None
    for c, val in enumerate(row, 1):
        cell = ws3.cell(r, c, val)
        cell.font = DATA_FONT
        cell.border = BORDER
        cell.alignment = Alignment(horizontal='left' if c == 1 else 'center')
        if alt_fill:
            cell.fill = alt_fill

ws3.column_dimensions['A'].width = 25
for i in range(2, len(pivot.columns) + 1):
    ws3.column_dimensions[get_column_letter(i)].width = 12
ws3.freeze_panes = "B2"

# ──────────────────── Sheet 4: Elbow Method ────────────────────
ws4 = wb.create_sheet("Elbow Method")
write_header(ws4, ['K (Jumlah Cluster)', 'Cost (Inertia)', 'Keterangan'])
for r, (k, c) in enumerate(sorted(elbow_costs.items()), 2):
    ws4.cell(r, 1, k).font = DATA_FONT
    ws4.cell(r, 2, round(c, 4)).font = DATA_FONT
    cell = ws4.cell(r, 3, "← K OPTIMAL TERPILIH" if k == K_OPT else "")
    cell.font = Font(name="Arial", size=10, bold=(k == K_OPT), color="C00000" if k == K_OPT else "000000")

for i, w in enumerate([22, 18, 25], 1):
    ws4.column_dimensions[get_column_letter(i)].width = w

# ──────────────────── Sheet 5: Status SK per Kecamatan ────────────────────
TODAY = pd.Timestamp.today().normalize()

df_clean['berakhir_sk_dt'] = pd.to_datetime(df_clean['berakhir_sk'], errors='coerce')
df_clean['sisa_hari']      = (df_clean['berakhir_sk_dt'] - TODAY).dt.days

cluster_mean_sisa = df_clean.groupby('cluster')['sisa_hari'].mean().sort_values()
cluster_urut      = list(cluster_mean_sisa.index)
n_cl              = len(cluster_urut)
tercile           = max(n_cl // 3, 1)

status_map = {}
for i, cl in enumerate(cluster_urut):
    if i < tercile:
        status_map[cl] = 'BERAKHIR'
    elif i < 2 * tercile:
        status_map[cl] = 'RENTAN'
    else:
        status_map[cl] = 'PANJANG'

df_clean['status_sk'] = df_clean['cluster'].map(status_map)

# Helper format tanggal dan sisa hari
def fmt_date(val):
    try:
        return pd.Timestamp(val).strftime('%Y-%m-%d')
    except Exception:
        return ''

def fmt_sisa(val):
    v = int(round(val))
    return f"Lewat {abs(v)} hari" if v < 0 else f"{v} hari"

# Agregasi per kecamatan
agg_kecamatan = df_clean.groupby('kecamatan').agg(
    jumlah_kendaraan  = ('kecamatan',    'count'),
    rata_rata_berakhir= ('berakhir_sk_dt','mean'),
    rata_rata_sisa_hari=('sisa_hari',     'mean'),
    min_berakhir      = ('berakhir_sk_dt','min'),
    max_berakhir      = ('berakhir_sk_dt','max'),
    jml_berakhir      = ('status_sk', lambda x: (x == 'BERAKHIR').sum()),
    jml_rentan        = ('status_sk', lambda x: (x == 'RENTAN').sum()),
    jml_panjang       = ('status_sk', lambda x: (x == 'PANJANG').sum()),
).reset_index().sort_values('rata_rata_sisa_hari').reset_index(drop=True)

# Agregasi per kode trayek
agg_trayek = df_clean.groupby('kode_trayek').agg(
    jumlah_kendaraan  = ('kode_trayek',  'count'),
    rata_rata_berakhir= ('berakhir_sk_dt','mean'),
    rata_rata_sisa_hari=('sisa_hari',     'mean'),
    min_berakhir      = ('berakhir_sk_dt','min'),
    max_berakhir      = ('berakhir_sk_dt','max'),
    jml_berakhir      = ('status_sk', lambda x: (x == 'BERAKHIR').sum()),
    jml_rentan        = ('status_sk', lambda x: (x == 'RENTAN').sum()),
    jml_panjang       = ('status_sk', lambda x: (x == 'PANJANG').sum()),
).reset_index().sort_values('rata_rata_sisa_hari').reset_index(drop=True)

# Warna baris per status (latar belakang ringan)
STATUS_ROW_FILL = {
    'BERAKHIR': PatternFill("solid", fgColor="FFF0F0"),
    'RENTAN':   PatternFill("solid", fgColor="FFF5E6"),
    'PANJANG':  PatternFill("solid", fgColor="F0FFF0"),
}

def status_baris(rata_sisa):
    """Tentukan status baris berdasarkan rata-rata sisa hari."""
    p33 = df_clean['sisa_hari'].quantile(0.33)
    p66 = df_clean['sisa_hari'].quantile(0.66)
    if rata_sisa <= p33:
        return 'BERAKHIR'
    elif rata_sisa <= p66:
        return 'RENTAN'
    else:
        return 'PANJANG'

ws5 = wb.create_sheet("Status SK per Kecamatan")

# Mapping cluster → status berdasarkan rata-rata berakhir_sk
# cluster dengan rata-rata berakhir_sk paling AWAL (kecil) = BERAKHIR
# cluster TENGAH = RENTAN, paling AKHIR (besar) = PANJANG
cluster_mean_berakhir = df_clean.groupby('cluster')['berakhir_sk_unix'].mean().sort_values()
cluster_urut = list(cluster_mean_berakhir.index)
n_cl = len(cluster_urut)
tercile = max(n_cl // 3, 1)

# Agregasi per kecamatan — ikuti pola kode kec_summary yang sudah ada
kec_status = df_clean.groupby('kecamatan').agg(
    cluster_dominan=('cluster',      lambda x: x.value_counts().index[0]),
    trayek_dominan=('kode_trayek',   lambda x: x.value_counts().index[0]),
    jumlah_kendaraan=('kecamatan',   'count'),
    rata_rata_berakhir=('berakhir_sk', 'mean'),
    min_berakhir=('berakhir_sk',     'min'),
    max_berakhir=('berakhir_sk',     'max'),
    jml_berakhir=('status_sk',       lambda x: (x == 'BERAKHIR').sum()),
    jml_rentan=('status_sk',         lambda x: (x == 'RENTAN').sum()),
    jml_panjang=('status_sk',        lambda x: (x == 'PANJANG').sum()),
    status_dominan=('status_sk',     lambda x: x.value_counts().index[0]),
).reset_index().sort_values('rata_rata_berakhir').reset_index(drop=True)

# Header Sheet 5
hdr5 = ['No', 'Kecamatan', 'Cluster', 'Trayek Dominan', 'Jumlah\nKendaraan',
        'Rata-rata\nBerakhir SK', 'Min Berakhir SK', 'Max Berakhir SK',
        'Jml\nBERAKHIR', 'Jml\nRENTAN', 'Jml\nPANJANG', 'STATUS SK']
write_header(ws5, hdr5)

for i, row_kec in enumerate(kec_status.itertuples(index=False), 1):
    r_excel  = i + 1
    cl       = row_kec[1]   # cluster_dominan
    status   = row_kec[10]  # status_dominan (index 10, setelah jml_panjang di index 9)
    row_fill = STATUS_ROW_FILL.get(status, PatternFill("solid", fgColor="F5F5F5"))
    cl_fill  = PatternFill("solid", fgColor=CLUSTER_COLORS[(cl - 1) % len(CLUSTER_COLORS)])
    vals = [
        i,
        row_kec[0],                                                                          # kecamatan
        cl,                                                                                  # cluster_dominan
        row_kec[2],                                                                          # trayek_dominan
        row_kec[3],                                                                          # jumlah_kendaraan
        row_kec[4].strftime('%Y-%m-%d') if isinstance(row_kec[4], pd.Timestamp) else '',    # rata_rata_berakhir
        row_kec[5].strftime('%Y-%m-%d') if isinstance(row_kec[5], pd.Timestamp) else '',    # min_berakhir
        row_kec[6].strftime('%Y-%m-%d') if isinstance(row_kec[6], pd.Timestamp) else '',    # max_berakhir
        row_kec[7],                                                                          # jml_berakhir
        row_kec[8],                                                                          # jml_rentan
        row_kec[9],                                                                          # jml_panjang
        status,                                                                              # status_dominan
    ]
    for c, val in enumerate(vals, 1):
        cell = ws5.cell(r_excel, c, val)
        cell.border    = BORDER
        cell.alignment = Alignment(horizontal='left' if c == 2 else 'center', vertical='center')
        if c == 3:   # kolom Cluster — warna cluster
            cell.fill = cl_fill
            cell.font = Font(name="Arial", size=10, bold=True, color="000000")
        elif c == 12:  # kolom Status SK — warna status
            cell.fill = STATUS_FILL.get(status, PatternFill("solid", fgColor="808080"))
            cell.font = STATUS_FONT.get(status, DATA_FONT)
        else:
            cell.fill = row_fill
            cell.font = DATA_FONT
    ws5.row_dimensions[r_excel].height = 18

for i, w in enumerate([5, 22, 10, 14, 10, 16, 14, 14, 10, 9, 9, 12], 1):
    ws5.column_dimensions[get_column_letter(i)].width = w
ws5.row_dimensions[1].height = 36
ws5.freeze_panes = "A2"


# ──────────────────── Sheet 6: Status SK per Kode Trayek ────────────────────
ws6 = wb.create_sheet("Status SK per Kode Trayek")

# Agregasi per kode trayek — pola sama seperti kec_status di atas
trayek_status = df_clean.groupby('kode_trayek').agg(
    cluster_dominan=('cluster',      lambda x: x.value_counts().index[0]),
    kecamatan_dominan=('kecamatan',  lambda x: x.value_counts().index[0]),
    jumlah_kendaraan=('kode_trayek', 'count'),
    rata_rata_berakhir=('berakhir_sk', 'mean'),
    min_berakhir=('berakhir_sk',     'min'),
    max_berakhir=('berakhir_sk',     'max'),
    jml_berakhir=('status_sk',       lambda x: (x == 'BERAKHIR').sum()),
    jml_rentan=('status_sk',         lambda x: (x == 'RENTAN').sum()),
    jml_panjang=('status_sk',        lambda x: (x == 'PANJANG').sum()),
    status_dominan=('status_sk',     lambda x: x.value_counts().index[0]),
).reset_index().sort_values('rata_rata_berakhir').reset_index(drop=True)

# Header Sheet 6
hdr6 = ['No', 'Kode Trayek', 'Cluster', 'Kecamatan Dominan', 'Jumlah\nKendaraan',
        'Rata-rata\nBerakhir SK', 'Min Berakhir SK', 'Max Berakhir SK',
        'Jml\nBERAKHIR', 'Jml\nRENTAN', 'Jml\nPANJANG', 'STATUS SK']
write_header(ws6, hdr6)

for i, row_tr in enumerate(trayek_status.itertuples(index=False), 1):
    r_excel  = i + 1
    cl       = row_tr[1]   # cluster_dominan
    status   = row_tr[10]  # status_dominan (index 10, setelah jml_panjang di index 9)
    row_fill = STATUS_ROW_FILL.get(status, PatternFill("solid", fgColor="F5F5F5"))
    cl_fill  = PatternFill("solid", fgColor=CLUSTER_COLORS[(cl - 1) % len(CLUSTER_COLORS)])
    vals = [
        i,
        row_tr[0],                                                                           # kode_trayek
        cl,                                                                                  # cluster_dominan
        row_tr[2],                                                                           # kecamatan_dominan
        row_tr[3],                                                                           # jumlah_kendaraan
        row_tr[4].strftime('%Y-%m-%d') if isinstance(row_tr[4], pd.Timestamp) else '',      # rata_rata_berakhir
        row_tr[5].strftime('%Y-%m-%d') if isinstance(row_tr[5], pd.Timestamp) else '',      # min_berakhir
        row_tr[6].strftime('%Y-%m-%d') if isinstance(row_tr[6], pd.Timestamp) else '',      # max_berakhir
        row_tr[7],                                                                           # jml_berakhir
        row_tr[8],                                                                           # jml_rentan
        row_tr[9],                                                                           # jml_panjang
        status,                                                                              # status_dominan
    ]
    for c, val in enumerate(vals, 1):
        cell = ws6.cell(r_excel, c, val)
        cell.border    = BORDER
        cell.alignment = Alignment(horizontal='center', vertical='center')
        if c == 3:   # kolom Cluster — warna cluster
            cell.fill = cl_fill
            cell.font = Font(name="Arial", size=10, bold=True, color="000000")
        elif c == 12:  # kolom Status SK — warna status
            cell.fill = STATUS_FILL.get(status, PatternFill("solid", fgColor="808080"))
            cell.font = STATUS_FONT.get(status, DATA_FONT)
        else:
            cell.fill = row_fill
            cell.font = DATA_FONT
    ws6.row_dimensions[r_excel].height = 18

for i, w in enumerate([5, 14, 10, 22, 10, 16, 14, 14, 10, 9, 9, 12], 1):
    ws6.column_dimensions[get_column_letter(i)].width = w
ws6.row_dimensions[1].height = 36
ws6.freeze_panes = "A2"

# ──────────────────── Sheet 7: "Evaluasi Clustering" ────────────────────
# Sheet ini menyajikan dua metode evaluasi:
# A. Elbow Method  → menentukan jumlah K yang optimal
# B. Pengaruh Fitur (Cramér's V & Eta²) → menjelaskan kontribusi tiap fitur

ws7 = wb.create_sheet("Evaluasi Clustering")

def tulis_sel(ws, row, col, nilai, bold=False, center=True,
              bg_color=None, font_color="000000", wrap=False):
    cell           = ws.cell(row, col, nilai)
    cell.font      = Font(name="Arial", size=10, bold=bold, color=font_color)
    cell.alignment = Alignment(
        horizontal='center' if center else 'left',
        vertical='center', wrap_text=wrap)
    cell.border = BORDER
    if bg_color:
        cell.fill = PatternFill("solid", fgColor=bg_color)
    return cell

def interpretasi_cramers(v):
    if v >= 0.50:   return "Kuat — fitur sangat menentukan pemisahan cluster"
    elif v >= 0.30: return "Moderat — fitur cukup berpengaruh"
    elif v >= 0.10: return "Lemah — fitur sedikit berpengaruh"
    else:           return "Sangat Lemah — hampir tidak berpengaruh"

def interpretasi_eta2(e):
    if e >= 0.50:   return "Sangat Kuat — cluster memisahkan nilai SK dengan baik"
    elif e >= 0.25: return "Moderat — ada perbedaan antar cluster namun ada overlap"
    elif e >= 0.06: return "Lemah — perbedaan antar cluster kecil"
    else:           return "Sangat Lemah — hampir tidak ada perbedaan"

baris = 1

# ── BAGIAN A: Elbow Method ────────────────────────────────────────
ws7.merge_cells(f'A{baris}:E{baris}')
tulis_sel(ws7, baris, 1, 'A. ELBOW METHOD — Pemilihan Jumlah Cluster Optimal (K)',
          bold=True, bg_color="1F4E79", font_color="FFFFFF")
ws7.row_dimensions[baris].height = 24

baris += 1
# Keterangan singkat tentang metode
ws7.merge_cells(f'A{baris}:E{baris}')
ket_elbow = ("Elbow Method memilih K optimal berdasarkan titik di mana "
             "penurunan nilai Cost (Inertia) paling signifikan terjadi — "
             "ibarat 'siku' pada grafik. K yang dipilih adalah titik "
             "di mana penambahan cluster berikutnya tidak lagi memberikan "
             "penurunan cost yang berarti.")
cell_ket           = ws7.cell(baris, 1, ket_elbow)
cell_ket.font      = Font(name="Arial", size=9, italic=True, color="404040")
cell_ket.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
cell_ket.border    = BORDER
cell_ket.fill      = PatternFill("solid", fgColor="EEF4FF")
ws7.row_dimensions[baris].height = 40

baris += 1
for col, teks in enumerate(['K (Jumlah Cluster)', 'Cost / Inertia',
                             'Selisih Penurunan', 'Keterangan', ''], 1):
    tulis_sel(ws7, baris, col, teks, bold=True, bg_color="2E75B6", font_color="FFFFFF")
ws7.row_dimensions[baris].height = 20

baris += 1
ks_sorted = sorted(elbow_costs.keys())
cost_vals = [elbow_costs[k] for k in ks_sorted]
for idx_k, k in enumerate(ks_sorted):
    selisih = (cost_vals[idx_k - 1] - cost_vals[idx_k]) if idx_k > 0 else '-'
    is_opt  = (k == K_OPT)
    bg      = "E2EFDA" if is_opt else None
    tulis_sel(ws7, baris, 1, k,                   bold=is_opt, bg_color=bg)
    tulis_sel(ws7, baris, 2, round(elbow_costs[k], 2), bold=is_opt, bg_color=bg)
    tulis_sel(ws7, baris, 3,
              round(selisih, 2) if selisih != '-' else '-', bold=is_opt, bg_color=bg)
    tulis_sel(ws7, baris, 4,
              f'← K={k} TERPILIH sebagai K Optimal' if is_opt else '',
              bold=is_opt, bg_color=bg,
              font_color="375623" if is_opt else "000000", center=False)
    tulis_sel(ws7, baris, 5, '', bg_color=bg)
    ws7.row_dimensions[baris].height = 18
    baris += 1

# ── BAGIAN B: Pengaruh Fitur ──────────────────────────────────────
baris += 1
ws7.merge_cells(f'A{baris}:E{baris}')
tulis_sel(ws7, baris, 1,
          "B. PENGARUH FITUR TERHADAP CLUSTER — Cramér's V & Eta²",
          bold=True, bg_color="1F4E79", font_color="FFFFFF")
ws7.row_dimensions[baris].height = 24

baris += 1
ws7.merge_cells(f'A{baris}:E{baris}')
ket_fitur = ("Cramér's V digunakan untuk mengukur kekuatan asosiasi antara "
             "fitur KATEGORIKAL (Kecamatan, Kode Trayek) dengan label cluster, "
             "diturunkan dari uji Chi-Square (rentang 0–1). "
             "Eta² digunakan untuk fitur NUMERIK (Berakhir SK), mengukur "
             "proporsi variansi yang dijelaskan oleh perbedaan antar cluster "
             "(rentang 0–1). Makin tinggi nilainya, makin besar kontribusi "
             "fitur tersebut dalam membentuk pemisahan cluster.")
cell_ket2           = ws7.cell(baris, 1, ket_fitur)
cell_ket2.font      = Font(name="Arial", size=9, italic=True, color="404040")
cell_ket2.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
cell_ket2.border    = BORDER
cell_ket2.fill      = PatternFill("solid", fgColor="EEF4FF")
ws7.row_dimensions[baris].height = 55

baris += 1
for col, teks in enumerate(["Peringkat", "Fitur", "Tipe Data",
                             "Ukuran Statistik", "Nilai", "Interpretasi"], 1):
    tulis_sel(ws7, baris, col, teks, bold=True, bg_color="2E75B6", font_color="FFFFFF")
ws7.row_dimensions[baris].height = 20

baris += 1
fitur_rows = [
    ("Kode Trayek", "Kategorikal", "Cramér's V", cramers_tray, interpretasi_cramers(cramers_tray)),
    ("Kecamatan",   "Kategorikal", "Cramér's V", cramers_kec,  interpretasi_cramers(cramers_kec)),
    ("Berakhir SK", "Numerik",     "Eta²",        eta2,         interpretasi_eta2(eta2)),
]
fitur_rows_sorted = sorted(fitur_rows, key=lambda x: x[3], reverse=True)

for rank, (fitur, tipe, ukuran, nilai, interp) in enumerate(fitur_rows_sorted, 1):
    is_top = (rank == 1)
    bg     = "FFF2CC" if is_top else None
    tulis_sel(ws7, baris, 1, f"{'★ ' if is_top else ''}{rank}",
              bold=is_top, bg_color=bg)
    tulis_sel(ws7, baris, 2, fitur,  bold=is_top, bg_color=bg, center=False)
    tulis_sel(ws7, baris, 3, tipe,   bg_color=bg)
    tulis_sel(ws7, baris, 4, ukuran, bg_color=bg)
    tulis_sel(ws7, baris, 5, round(nilai, 4), bold=is_top, bg_color=bg)
    tulis_sel(ws7, baris, 6, interp, bg_color=bg, center=False)
    ws7.row_dimensions[baris].height = 18
    baris += 1

# ── BAGIAN C: Kesimpulan Evaluasi ─────────────────────────────────
baris += 1
ws7.merge_cells(f'A{baris}:F{baris}')
tulis_sel(ws7, baris, 1, 'C. KESIMPULAN EVALUASI',
          bold=True, bg_color="1F4E79", font_color="FFFFFF")
ws7.row_dimensions[baris].height = 24

baris += 1
ws7.merge_cells(f'A{baris}:F{baris}')
kesimpulan = (
    f"Penentuan jumlah cluster optimal menggunakan Elbow Method menghasilkan K={K_OPT}, "
    f"yaitu nilai K di mana penurunan Cost/Inertia paling signifikan terjadi sehingga "
    f"penambahan cluster selanjutnya tidak lagi memberikan pemisahan yang berarti. "
    f"Analisis pengaruh fitur menunjukkan bahwa {fitur_terkuat} merupakan fitur paling "
    f"dominan dalam pembentukan cluster dengan skor {skor_fitur[fitur_terkuat]:.4f}, "
    f"diikuti oleh {[f for f in skor_fitur if f != fitur_terkuat][0]} "
    f"({skor_fitur[[f for f in skor_fitur if f != fitur_terkuat][0]]:.4f}), "
    f"dan {[f for f in skor_fitur if f != fitur_terkuat][1]} "
    f"({skor_fitur[[f for f in skor_fitur if f != fitur_terkuat][1]]:.4f}). "
    f"Hal ini mengindikasikan bahwa pengelompokan kendaraan lebih banyak dipengaruhi "
    f"oleh kesamaan rute trayek dan wilayah kecamatan dibandingkan nilai timestamp "
    f"masa berlaku SK semata."
)
cell_kes           = ws7.cell(baris, 1, kesimpulan)
cell_kes.font      = Font(name="Arial", size=10, italic=True)
cell_kes.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
cell_kes.border    = BORDER
cell_kes.fill      = PatternFill("solid", fgColor="F2F2F2")
ws7.row_dimensions[baris].height = 80

# Lebar kolom Sheet 7
for i, w in enumerate([10, 18, 14, 16, 10, 45], 1):
    ws7.column_dimensions[get_column_letter(i)].width = w
ws7.freeze_panes = "A2"

print("   ✓ Sheet 'Evaluasi Clustering' berhasil dibuat")

wb.save(OUTPUT_FILE)
print(f"\n{'='*60}")
print(f" SELESAI!")
print(f" Output : {OUTPUT_FILE}")
print(f" Data   : {len(df_clean)} baris")
print(f" Cluster: {K_OPT}")
print(f"{'='*60}")


[6] EXPORT KE EXCEL ...
   ✓ Sheet 'Evaluasi Clustering' berhasil dibuat

 SELESAI!
 Output : /content/drive/MyDrive/KULIAH/Semester 8/SKRIPSI-TugasAkhir/SKRIPSI /PENGOLAHAN DATA/hasil_clustering_kprototype.xlsx
 Data   : 6017 baris
 Cluster: 3


In [8]:
# ─── VERIFIKASI ANGKA UNTUK SKRIPSI ───────────────────────────
TODAY = pd.Timestamp.today().normalize()
THRESHOLD = TODAY + pd.DateOffset(years=1)

print(f"Tanggal Analisis     : {TODAY.strftime('%d %B %Y')}")
print(f"Threshold SK Aktif   : {THRESHOLD.strftime('%d %B %Y')}")
print("=" * 55)

for cl in sorted(df_clean['cluster'].unique()):
    subset = df_clean[df_clean['cluster'] == cl]
    aktif_panjang = (subset['berakhir_sk_dt'] > THRESHOLD).sum()
    pct = aktif_panjang / len(subset) * 100
    print(f"Cluster {cl} : {aktif_panjang} kendaraan ({pct:.1f}%) SK aktif > 1 tahun")

print("=" * 55)

Tanggal Analisis     : 29 April 2026
Threshold SK Aktif   : 29 April 2027
Cluster 1 : 543 kendaraan (24.3%) SK aktif > 1 tahun
Cluster 2 : 149 kendaraan (6.1%) SK aktif > 1 tahun
Cluster 3 : 46 kendaraan (3.4%) SK aktif > 1 tahun
